**Where the data comes from.** `results/GA/run_N` is written by `python AI/GA/optimize.py` (headless simulator; `Time` = virtual game seconds = pieces x ms-per-piece; fitness = mean game score + 1 per piece; each run also logs `Deaths`, `AvgPieces`, `AvgLinesPerSec` and a baseline agent (`BaselineFitness`, `RelBest`, `RelAvg`)). The cells below show the newest run; to look at another one, pass `GA_DIR / "run_3" / "ga_log.csv"` (or `agent_log.csv`) instead of `latest_run(GA_DIR)`. The functions also still read the old real-time format.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# --- project paths (work from any location) ---
from pathlib import Path


def find_project_root(start=None):
    """Walk up from the notebook's folder until the folder containing AI/ and Tetris/ is found."""
    here = Path(start or Path.cwd()).resolve()
    for d in [here, *here.parents]:
        if (d / "AI").is_dir() and (d / "Tetris").is_dir():
            return d
    raise FileNotFoundError("Open this notebook from inside the Tetris-Project folder.")


ROOT = find_project_root()
GA_DIR = ROOT / "results" / "GA"
GAUNTLET_DIR = ROOT / "results" / "gauntlet"


def latest_run(base_dir):
    """Highest-numbered run_N folder inside base_dir."""
    runs = [d for d in Path(base_dir).glob("run_*") if d.name[4:].isdigit()]
    if not runs:
        raise FileNotFoundError(f"No run_N folders in {base_dir}")
    return max(runs, key=lambda d: int(d.name[4:]))


def plot_ga_run(csv_path, run_label=None, save=False):
    # Style and font config for professional plots
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.rcParams.update({
        "font.size": 13,
        "axes.labelsize": 14,
        "axes.titlesize": 15,
        "legend.fontsize": 11,
        "figure.dpi": 150,
        "axes.edgecolor": "#222222",
        "axes.linewidth": 1.2,
        "axes.grid": True,
        "grid.alpha": 0.18,
        "xtick.direction": "out",
        "ytick.direction": "out"
    })

    df = pd.read_csv(csv_path)
    if run_label is None:
        run_label = Path(csv_path).parent.name

    # Use constrained_layout for clean, warning-free grid
    fig, axs = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
    fig.suptitle(f"GA Training Summary — {run_label}", fontsize=18, fontweight='bold', y=1.02)

    # 1. Fitness Progress
    axs[0, 0].plot(df["Generation"], df["BestFitness"], label="Best Fitness", color="#1752a6", lw=2)
    axs[0, 0].plot(df["Generation"], df["AvgFitness"], label="Average Fitness", color="#d18300", lw=1.7, linestyle="--")
    axs[0, 0].plot(df["Generation"], df["WorstFitness"], label="Worst Fitness", color="#c02d2d", lw=1, linestyle=":")
    axs[0, 0].set_xlabel("Generation")
    axs[0, 0].set_ylabel("Fitness")
    axs[0, 0].set_title("Fitness Progress")
    axs[0, 0].legend(loc="best", frameon=True)
    axs[0, 0].spines['top'].set_visible(False)
    axs[0, 0].spines['right'].set_visible(False)

    # 2. Fitness Std Dev
    axs[0, 1].plot(df["Generation"], df["FitnessStd"], label="Fitness Std. Dev.", color="#735da5", lw=2)
    axs[0, 1].set_xlabel("Generation")
    axs[0, 1].set_ylabel("Std. Dev.")
    axs[0, 1].set_title("Fitness Std. Deviation")
    axs[0, 1].legend(loc="best", frameon=True)
    axs[0, 1].spines['top'].set_visible(False)
    axs[0, 1].spines['right'].set_visible(False)

    # 3. Best Score (skip if not present)
    if "BestScore" in df.columns:
        axs[0, 2].plot(df["Generation"], df["BestScore"], color="#2b6e3a", lw=2)
        axs[0, 2].set_title("Best Agent Score")
        axs[0, 2].set_xlabel("Generation")
        axs[0, 2].set_ylabel("Score")
        axs[0, 2].spines['top'].set_visible(False)
        axs[0, 2].spines['right'].set_visible(False)
    else:
        axs[0, 2].set_title("No Score Data")
        axs[0, 2].axis('off')

    # 4. Best Lines (skip if not present)
    if "BestLines" in df.columns:
        axs[1, 0].plot(df["Generation"], df["BestLines"], color="#ba5d12", lw=2)
        axs[1, 0].set_title("Best Agent Lines Cleared")
        axs[1, 0].set_xlabel("Generation")
        axs[1, 0].set_ylabel("Lines")
        axs[1, 0].spines['top'].set_visible(False)
        axs[1, 0].spines['right'].set_visible(False)
    else:
        axs[1, 0].set_title("No Lines Data")
        axs[1, 0].axis('off')

    # 5. NumTetris (skip if not present)
    if "NumTetris" in df.columns:
        axs[1, 1].plot(df["Generation"], df["NumTetris"], color="#c02d2d", lw=2)
        axs[1, 1].set_title("Best Agent Tetrises")
        axs[1, 1].set_xlabel("Generation")
        axs[1, 1].set_ylabel("Tetrises")
        axs[1, 1].spines['top'].set_visible(False)
        axs[1, 1].spines['right'].set_visible(False)
    else:
        axs[1, 1].set_title("No Tetris Data")
        axs[1, 1].axis('off')

    # 6. Heuristic Weights Evolution
    param_cols = [
        "W1:AggHeight", "W2:Holes", "W3:Blockades", "W4:Bumpiness", "W5:AlmostFull",
        "W6:FillsWell", "W7:ClearBonus4", "W8:ClearBonus3", "W9:ClearBonus2", "W10:ClearBonus1"
    ]
    # Use matplotlib.colormaps to avoid deprecation warnings
    param_colors = plt.colormaps["tab10"]
    for idx, col in enumerate(param_cols):
        if col in df.columns:
            axs[1, 2].plot(df["Generation"], df[col], label=col.replace(":", ": "), color=param_colors(idx), lw=1.3)
    axs[1, 2].set_xlabel("Generation")
    axs[1, 2].set_ylabel("Weight Value")
    axs[1, 2].set_title("Best Agent Heuristic Weights")
    axs[1, 2].legend(loc="upper right", fontsize=8, frameon=True, ncol=1)
    axs[1, 2].spines['top'].set_visible(False)
    axs[1, 2].spines['right'].set_visible(False)

    # Clean up all axes for consistency
    for ax in axs.flat:
        ax.tick_params(axis='both', which='major', labelsize=12)

    plt.show()

    if save:
        fig.savefig(f"{run_label}_GA_Report.png", bbox_inches="tight", dpi=300)
        print(f"Saved as {run_label}_GA_Report.png")

# Example usage:
# plot_ga_run(GA_DIR / "run_1" / "ga_log.csv", run_label="Run 1")


def plot_ga_vs_baseline(csv_path, run_label=None):
    """
    Progress relative to the baseline (reference) agent.

    The baseline plays the SAME piece sequences as the population every generation, so
    "agent score / baseline score" cancels out generations whose sequences happened to be
    easy or hard. Only available for runs that logged the baseline columns.
    """
    df = pd.read_csv(csv_path)
    cols = ["BaselineFitness", "RelBest", "RelAvg"]
    if not all(c in df.columns for c in cols):
        print(f"{csv_path}: no baseline columns (older run, or started with --no-baseline).")
        return
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if run_label is None:
        run_label = Path(csv_path).parent.name

    fig, axs = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    fig.suptitle(f"{run_label} - progress vs. baseline agent", fontsize=16, fontweight="bold")

    axs[0].plot(df["Generation"], df["BaselineFitness"], color="#555555", lw=2, ls="--", label="Baseline (default weights)")
    axs[0].plot(df["Generation"], df["BestFitness"], color="#1752a6", lw=2, label="Best agent")
    axs[0].plot(df["Generation"], df["AvgFitness"], color="#d18300", lw=1.7, label="Population average")
    axs[0].set_title("Raw fitness (bounces with piece-sequence luck)")
    axs[0].set_xlabel("Generation")
    axs[0].set_ylabel("Fitness")
    axs[0].legend()

    axs[1].axhline(1.0, color="#555555", lw=1.5, ls="--", label="Baseline = 1.0")
    axs[1].plot(df["Generation"], df["RelBest"], color="#1752a6", lw=2, label="Best agent / baseline")
    axs[1].plot(df["Generation"], df["RelAvg"], color="#d18300", lw=1.7, label="Population average / baseline")
    axs[1].set_title("Relative to baseline (luck cancelled out)")
    axs[1].set_xlabel("Generation")
    axs[1].set_ylabel("Fitness / baseline fitness")
    axs[1].legend()

    for ax in axs:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    plt.show()


In [ ]:
# Newest training run (results/GA/run_N with the highest N)
plot_ga_run(latest_run(GA_DIR) / "ga_log.csv")


In [ ]:
# Same run, relative to the baseline agent (only runs that logged a baseline)
plot_ga_vs_baseline(latest_run(GA_DIR) / "ga_log.csv")
